In [1]:
# @title Run this cell => Restart the session => Start executing the below cells **(DO NOT EXECUTE THIS CELL AGAIN)**

!pip install -q langchain==0.3.21 \
                huggingface_hub==0.29.3 \
                openai==1.68.2 \
                chromadb==0.6.3 \
                langchain-community==0.3.20 \
                langchain_openai==0.3.10 \
                lark==1.2.2\
                rank_bm25==0.2.2\
                numpy==2.2.4 \
                scipy==1.15.2 \
                scikit-learn==1.6.1 \
                transformers==4.50.0 \
                pypdf==5.4.0 \
                markdown-pdf==1.7 \
                sentence_transformers==4.0.0 \
                torch==2.6.0+cu124


%pip install --upgrade chromadb
%pip install pillow
%pip install open-clip-torch
%pip install matplotlib

ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.6.2 Requires-Python >=3.7,<3.10; 1.6.3 Requires-Python >=3.7,<3.10; 1.7.0 Requires-Python >=3.7,<3.10; 1.7.1 Requires-Python >=3.7,<3.10; 1.7.2 Requires-Python >=3.7,<3.11; 1.7.3 Requires-Python >=3.7,<3.11; 1.8.0 Requires-Python >=3.8,<3.11; 1.8.0rc1 Requires-Python >=3.8,<3.11; 1.8.0rc2 Requires-Python >=3.8,<3.11; 1.8.0rc3 Requires-Python >=3.8,<3.11; 1.8.0rc4 Requires-Python >=3.8,<3.11; 1.8.1 Requires-Python >=3.8,<3.11
ERROR: Could not find a version that satisfies the requirement torch==2.6.0+cu124 (from versions: 2.0.0, 2.0.1, 2.1.0, 2.1.1, 2.1.2, 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0)
ERROR: No matching distribution found for torch==2.6.0+cu124


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Research Files and Config Files Loading

In [1]:
# Unzipping the Research Papers
import zipfile
with zipfile.ZipFile("Papers-20250513T175348Z-1-001.zip", 'r') as zip_ref:
  zip_ref.extractall("Papers/")

In [2]:
# @title Loading the `config.json` file
import json
import os

# Load the JSON file and extract values
file_name = 'config.json'
with open(file_name, 'r') as file:
    config = json.load(file)
    os.environ['AZURE_OPENAI_API_KEY'] = config.get("API_KEY") # Loading the API Key
    os.environ["AZURE_OPENAI_ENDPOINT"] = config.get("OPENAI_API_BASE") # Loading the API Base Url
    os.environ['OPENAI_API_VERSION'] = "2024-12-01-preview"


In [3]:
# @title Defining the Embedding Model - Using `text-embedding-ada-002` Model
from langchain_openai import AzureOpenAIEmbeddings
embeddings = AzureOpenAIEmbeddings(model="text-embedding-ada-002", show_progress_bar=True, max_retries=20, chunk_size=750)

In [4]:
# @title Defining the LLM Model - Using `gpt-4o-mini` Model
from langchain_openai import AzureChatOpenAI
llm = AzureChatOpenAI(model="gpt-4o-mini", temperature=0)

In [5]:
# Uploading multiple pdfs:
from glob import glob
from langchain_community.document_loaders import PyPDFLoader

# Path to folder with PDFs
DOC_FOLDER = "Papers/"
pdf_files = glob(DOC_FOLDER + "*.pdf")  # grabs all PDFs in the folder

all_pages = []

# Load each PDF and extract pages
for pdf_path in pdf_files:
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    all_pages.extend(pages)

Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 51 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 83 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 48 0 (offset 0)
Ignoring wrong pointing object 92 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong p

## Creating Questions for Each Document to Enhance Retrieval

In [7]:
hypothetical_questions_prompt = """
Generate a list of exactly 3 hypothetical questions that the below document could be used to answer:
{doc}
Generate only a list of questions. Do not mention anything before or after the list.
"""

In [12]:
from langchain_core.documents import Document
from tqdm import tqdm  # Import the tqdm library for the progress bar

hypothetical_questions_docs = []

for document in tqdm(all_pages, total=len(all_pages), desc='Processing documents'):
    try:
        response = llm.invoke(hypothetical_questions_prompt.format(doc = document))
        questions = response.content
    except Exception as e:
        print(e)
        questions = ""

    questions_metadata = {
        'parent_chunk_id': document.id,
        'parent_chunk': document.page_content
    }

    hypothetical_questions_docs.append(
        Document(
            id=document.id,
            page_content=questions,
            metadata=questions_metadata
        )
    )

Processing documents:  63%|██████▎   | 1140/1813 [20:45<11:16,  1.00s/it]

Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': True, 'severity': 'medium'}}}}}


Processing documents: 100%|██████████| 1813/1813 [32:22<00:00,  1.07s/it]


## Embedding Vectorstore

In [13]:
from langchain_chroma import Chroma

hypo_question_vectorstore = Chroma(
    collection_name="hypothetical_questions",
    embedding_function=embeddings
)

In [14]:
hypo_question_vectorstore.add_documents(
    ids = [d.id for d in hypothetical_questions_docs],
    documents=hypothetical_questions_docs
)

hypo_question_ret = hypo_question_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={'k':15}
)

  0%|          | 0/3 [00:00<?, ?it/s]

## Reranker

In [16]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
crossencoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

from langchain.retrievers.document_compressors import CrossEncoderReranker
reranker = CrossEncoderReranker(model=crossencoder, top_n=15)

from langchain.retrievers import ContextualCompressionRetriever
reranker_retriever = ContextualCompressionRetriever(
    base_compressor=reranker, base_retriever=hypo_question_ret
)

c:\Users\Pheon\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Loading the Notice of Funding Opportunity Document

In [17]:
from pypdf import PdfReader

NOFO_document = "PAR-25-136_ Laboratories to Optimize Digital Health (R0….pdf"
pdf_reader = PdfReader(NOFO_document)
NOFO_text = ""
for page in pdf_reader.pages:
    NOFO_text += page.extract_text() + '\n'
#text_data = pdf_loader.load()
#print(len(text_data))

Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 69 0 (offset 0)
Ignoring wrong pointing object 90 0 (offset 0)


In [18]:
print(type(NOFO_text[0]))

<class 'str'>


## Summarizing NOFO Document to Focus Retrieval

In [19]:
prompt = f"""
You are an expert AI assistant for parsing Notice of Funding Opportunity Documents and extracting relevant information necessary for writing a grant proposal document.
Such useful information includes but is not limited to: Application Instructions, Notice of Funding Opportuniy Descriptions, and Application and Submission Information

### Document:
{NOFO_text}
"""
llm_response = llm.invoke(prompt)
print("Answer: \n", llm_response.content)

Answer: 
 Here is a structured summary of the relevant information extracted from the Notice of Funding Opportunity (NOFO) document for the "Laboratories to Optimize Digital Health (R01 Clinical Trial Required)" funding opportunity:

### 1. Overview Information
- **Funding Opportunity Title**: Laboratories to Optimize Digital Health (R01 Clinical Trial Required)
- **Funding Opportunity Number (FON)**: PAR-25-136
- **Participating Organization(s)**: National Institutes of Health (NIH), National Institute of Mental Health (NIMH)
- **Posted Date**: November 07, 2024
- **Open Date (Earliest Submission Date)**: January 05, 2025
- **Expiration Date**: January 08, 2028

### 2. Key Dates
- **Application Due Dates**:
  - February 05, 2025
  - June 05, 2025
  - October 05, 2025
  - February 05, 2026
  - June 05, 2026
  - October 05, 2026
  - February 05, 2027
  - June 05, 2027
  - October 05, 2027
- **Earliest Start Date**: July 2025 (for applications submitted by February 05, 2025)

### 3. Fund

In [20]:
reranked_responses = reranker_retriever.get_relevant_documents(llm_response.content)
print(len(reranked_responses))

C:\Users\Pheon\AppData\Local\Temp\ipykernel_25268\405499254.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  reranked_responses = reranker_retriever.get_relevant_documents(llm_response.content)


  0%|          | 0/1 [00:00<?, ?it/s]

15


In [21]:
for document in reranked_responses:
    print(document)
    print('=' * 50)

page_content='1. How might the findings of the proposed study influence the design of HIV prevention campaigns in local health departments?
2. What are the implications of the study's funding by the National Institutes of Health for its credibility and potential impact on public health initiatives?
3. When will the data from the randomized controlled trial be available for public access, and under what conditions can it be requested?' metadata={'parent_chunk': 'JMIR Preprints Yang et al\nIn  summary,  f indings  of  this  proposed  study  will  transform  the  design,  evaluation,  and\nimplementation of HIV campaigns, potentially bringing new ideas for local health departments and\ncommunity-based organizations in developing more impactful PrEP campaigns.\nAcknowledgments\nThis study is funded by  the National Institutes of Health (National Institute of Mental Health:\nR34MH116725).  The content is solely the responsibility of the authors and does not necessarily\nrepresent the offici

In [ ]:
# Retrieve parent chunks from questions.
parent_chunks = []

for document in reranked_responses:
    parent_chunks.append(document.metadata['parent_chunk'])

print(parent_chunks)

['JMIR Preprints Yang et al\nIn  summary,  f indings  of  this  proposed  study  will  transform  the  design,  evaluation,  and\nimplementation of HIV campaigns, potentially bringing new ideas for local health departments and\ncommunity-based organizations in developing more impactful PrEP campaigns.\nAcknowledgments\nThis study is funded by  the National Institutes of Health (National Institute of Mental Health:\nR34MH116725).  The content is solely the responsibility of the authors and does not necessarily\nrepresent the official views of the NIH or the US government. The funding body did not participate\nin designing the study or in writing this manuscript.\nData Availability\nData sharing is not applicable to this study because data collection has not been completed or\nanalyzed yet for the preparation of this protocol manuscript. After analysis and dissemination, data\ngenerated and analyzed in the randomized controlled trial will be available from the corresponding\nauthor upon 

In [61]:
prompt = f"""
You are an expert AI assistant for writing grant proposal documents based off of previously written proposals.
Use the following pieces of retrieved Proposals to write a professional and high quality proposal that meets the requirements of the following Notice of Funding Opportunity document, including unique and specific details found within.

The drafted response should contain the following categories:
 - Introduction: A background on the overall problem.
 - The Problem Statement: What questions regarding the problem this research is targeting.  
 - Objectives: What parts of the problem is this proposed research is attempting to solve.
 - Preliminary Literature Review: A review of previous studies on this project.
 - Methodology: How the research plans to meet the stated objectives. This includes the following subsections:
     - Data Collection: How the data will be collected
     - Analysis: How the data will be analyzed
     - Ethical Considerations: How this research plans to meet ethical guidelines.
     - Timeline: How long each part of the research will take.
 - Budget Allocation: How much money the research will cost, how long the money is needed for, and how it is planned to be spent.
 - Conclusion

### Retreived Proposals:
{parent_chunks}

### Notice of Funding Opportunity Document:
{NOFO_text}
"""

In [62]:
llm_response = llm.invoke(prompt)

In [ ]:
print("Answer: \n", llm_response.content)

Answer: 
 # Grant Proposal: Optimizing Digital Health Interventions for Mental Health Outcomes

## Introduction
The landscape of mental health care in the United States is fraught with challenges, including significant treatment gaps and disparities in access to care. Approximately 50% of the U.S. population meets the criteria for a mental disorder at some point in their lives, yet only about 66% of individuals with serious mental health disorders receive treatment annually. This gap is exacerbated for marginalized populations, including racial and ethnic minorities, who often face additional barriers to accessing mental health services. Digital health technologies, including mobile health (mHealth) applications and online platforms, present a promising avenue to bridge these gaps by providing scalable, accessible, and evidence-based interventions. However, the effectiveness of these digital interventions remains underexplored, particularly in terms of their ability to engage users and

## AI Request Draft Review

In [86]:
ai_review_prompt = """
Your task is to judge the faithfulness of a document draft for a given Notice of Funding Opportunity (NOFO) Document based on a given context.
For each entire section of the draft, you must return verdict as 1 if the body of the section can be directly inferred based on context or NOFO document,
If the section can not be directly inferred from either the context or NOFO document, return 0.

Output Format:
Arrange your output in the following JSON format:
{
    "section": section from the draft
    "explanation": a step by step evaluation for faithfullness
    "rating": either 1 or 0
}
DO NOT output anything else before or after the JSON output.
"""

ai_review_draft_context = f"""
## Draft:
{llm_response.content}
## Notice of Funding Opportunity Document:
{NOFO_text}
## Context: 
{parent_chunks}
"""

In [87]:
faithfullness_review = llm.invoke(ai_review_prompt + ai_review_draft_context)
print(faithfullness_review.content)

```json
[
    {
        "section": "Introduction",
        "explanation": "The introduction discusses the challenges in mental health care in the U.S., including treatment gaps and disparities, which aligns with the context provided in the NOFO about the need for innovative research in digital mental health interventions. The mention of digital health technologies as a solution is also supported by the NOFO's emphasis on leveraging established platforms for mental health interventions.",
        "rating": 1
    },
    {
        "section": "The Problem Statement",
        "explanation": "The problem statement outlines specific research questions that align with the goals of the NOFO, which seeks to test strategies to optimize digital mental health interventions. The focus on user engagement and measuring impact on mental health outcomes is directly related to the objectives of the funding opportunity.",
        "rating": 1
    },
    {
        "section": "Objectives",
        "explanati

In [90]:
context_review_prompt = """
Your task is to judge the whether or not either the given context or Notice of Funding Opportunity (NOFO) document was useful in creating the given drafted document.
For each entire section of the draft, you must return verdict as 1 if either the context or NOFO document was useful in arriving at the section and a 0 if neither the context nor NOFO document were useful.

Output Format:
Arrange your output in the following JSON format:
{
    "section": section from the draft
    "explanation": a step by step evaluation for whether or not the context was useful
    "rating": either 1 or 0
}
DO NOT output anything else before or after the JSON output.
"""

context_review_draft_context = f"""
## Draft:
{llm_response.content}
## Notice of Funding Opportunity Document:
{NOFO_text}
## Context: 
{parent_chunks}
"""

In [91]:
context_review = llm.invoke(context_review_prompt + context_review_draft_context)
print(context_review.content)

```json
[
    {
        "section": "Introduction",
        "explanation": "The introduction discusses the challenges in mental health care and the potential of digital health technologies, which aligns with the NOFO's emphasis on optimizing digital mental health interventions. The statistics and context provided in the introduction are relevant to the funding opportunity's goals.",
        "rating": 1
    },
    {
        "section": "The Problem Statement",
        "explanation": "The problem statement outlines specific research questions that directly relate to the objectives of the NOFO, which seeks to test strategies for optimizing digital mental health interventions. This alignment indicates that the NOFO was useful in formulating these questions.",
        "rating": 1
    },
    {
        "section": "Objectives",
        "explanation": "The objectives listed in the draft are focused on developing and testing digital mental health interventions, which is a core purpose of the NOFO.